# 10 - Probe <-> causal tracing, TF-IDF (GPU, ~1-1.5 h)

**Settings:** GPU T4 x1 | Internet On | Secret `HF_TOKEN` (optional). Run after notebook 09 (separate session is fine).

The mid-term probe-vs-tracing tests (cell7_score_probes.py), on the same 219 TF-IDF pairs, for the **old** (C=1, block 10) and the **tuned** probe (C=0.01, block 12):
* **Test A**: the clean run's own hop 1 is teacher-forced after the clean and the CF prompt (same words, only the question entity differs). Does the probe still flag the CF?
* **Test B**: patch the entity (last token / all tokens / third-entity control) at block l; does the probe's score drop track the behavioural recovery (Spearman, blocks below the probe's layer)?

Features are saved (`features.npz`, ~200 MB) so robustness and grounding-direction analyses run on CPU afterwards. **Preflight:** `SELF-CHECK block 10/12: cosine ... = 1.00` (must be >= 0.99, it aborts otherwise). OOM: lower `--batch`.

Download `probe_tracing_tfidf.zip`.

In [ ]:
# ===== Setup: internet, clone, HF token, GPU =====
import os, socket, subprocess, sys, time, zipfile, json
for host in ('github.com', 'huggingface.co'):
    socket.gethostbyname(host)
WORK = '/kaggle/working/LinguaFranca'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'cf-tfidf', 'https://github.com/ANLPproject/LinguaFranca.git', WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull'], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')    # read by the scripts from the environment
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print(f'No HF_TOKEN secret ({e.__class__.__name__}); the ungated model mirror will be used')
import torch
assert torch.cuda.is_available(), 'No GPU: Session options -> Accelerator -> GPU T4 x1, then restart.'

def run(cmd):
    '''Stream a command's output; return its exit code.'''
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        if 'it/s]' in line or 's/it]' in line:
            continue
        print(line, end='')
    return p.wait()

OUT = '/kaggle/working/probe_tracing_tfidf'
PRE = '/kaggle/working/probe_tracing_preflight'

In [ ]:
# ===== PREFLIGHT (2 pairs) =====
rc = run([sys.executable, 'probe_tracing.py', '--out', PRE, '--max-pairs', '2'])
assert rc == 0 and os.path.exists(f'{PRE}/summary.md'), 'PREFLIGHT FAILED - read the log above.'
print('PREFLIGHT PASSED')

In [ ]:
# ===== MAIN RUN: the 219 pairs of notebooks 05/08 (resumable) =====
rc = run([sys.executable, 'probe_tracing.py', '--out', OUT, '--max-pairs', '219'])
print('exit code:', rc)

In [ ]:
# ===== package (always run) =====
zpath = '/kaggle/working/probe_tracing_tfidf.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in (OUT, PRE):
        for root, _, files in os.walk(d):
            for f in files:
                p = os.path.join(root, f)
                z.write(p, os.path.relpath(p, os.path.dirname(d)))
print(f'{zpath}: {os.path.getsize(zpath) / 1e6:.1f} MB')